# RQ2 - Du bao AR vs Random Walk

Muc tieu: kiem tra nang luc du bao noi tai cua Level/Slope/Curvature bang AR(p), so sanh voi Random Walk qua nhieu horizon.

In [1]:
import sys
sys.path.append('..')

from src import config, forecasting as fc
import pandas as pd

## 1. Nap factor tu RQ1

In [2]:
from src.utils import ensure_monthly_freq

factors_monthly = pd.read_csv('../data/ns_factors_monthly.csv', index_col=0, parse_dates=True)
factors_monthly = ensure_monthly_freq(factors_monthly)
factors_monthly.tail()

,level,slope,curvature
date,,,
2026-05,5.019246,-1.235283,-1.775704
2026-06,4.895557,-1.021234,-1.172478
2026-07,5.286283,-1.419788,-1.324546
2026-08,5.247621,-1.330492,-1.128791
2026-09,5.497331,-1.401444,-0.148833


## 2. Chay pipeline AR vs RW cho tung nhan to

In [3]:
results = []
for col in ['level', 'slope', 'curvature']:
    print(f'Running AR vs RW for {col} ...')
    res = fc.run_ar_vs_rw_pipeline(factors_monthly, col)
    print(f'  -> {len(res)} horizons')
    results.append(res)

summary = pd.concat(results, ignore_index=True)
summary

Running AR vs RW for level ...
  -> 4 horizons
Running AR vs RW for slope ...
  -> 4 horizons
Running AR vs RW for curvature ...
  -> 4 horizons


,factor,horizon,ar_lag,RMSE_AR,RMSE_RW,MAE_AR,MAE_RW,n_obs,DM_stat,DM_pvalue
0,level,1,1,0.230802,0.228743,0.183030,0.181566,129,0.877141,0.382053
1,level,3,1,0.415918,0.408680,0.321538,0.307709,127,0.690474,0.491166
2,level,6,1,0.567871,0.549869,0.460723,0.431648,124,0.636353,0.525729
3,level,12,1,0.784076,0.760011,0.645228,0.620488,118,0.334655,0.738485
4,slope,1,1,0.311246,0.308662,0.225511,0.226482,129,0.746376,0.456809
5,slope,3,1,0.565752,0.555644,0.429369,0.415861,127,0.700083,0.485166
6,slope,6,1,0.855538,0.837223,0.640863,0.643456,124,0.468448,0.640294
7,slope,12,1,1.358363,1.346475,1.021786,1.060653,118,0.126939,0.899207
8,curvature,1,1,0.698008,0.699661,0.510038,0.511922,129,-0.126484,0.899548
9,curvature,3,1,1.164599,1.177612,0.880228,0.880443,127,-0.297409,0.766644


## 3. Doc ket qua

- `rolling=True`: expanding-window OOS (train <= 2023-12, test tu 2024-01).
- AR tot hon RW ve diem so neu `RMSE_AR < RMSE_RW` (hoac MAE).
- Diebold-Mariano: `DM_pvalue < 0.05` => chenh lech **co y nghia thong ke**.
- Neu bang `DM_pvalue < 0.05` **rong**: khong sai data — nghia la tren mau test ngan (~22-33 origin) AR chua vuot RW mot cach thong ke, du RMSE co the thap hon.

In [4]:
# 1) So sanh diem so (khong can p-value)
rmse_win = summary.assign(
    AR_better_RMSE=summary['RMSE_AR'] < summary['RMSE_RW'],
    MAE_better=summary['MAE_AR'] < summary['MAE_RW'],
)[['factor', 'horizon', 'RMSE_AR', 'RMSE_RW', 'AR_better_RMSE', 'MAE_AR', 'MAE_RW', 'MAE_better', 'DM_pvalue']]
print('=== AR vs RW (RMSE/MAE) ===')
display(rmse_win)

# 2) Chi cac dong DM co y nghia thong ke
sig = summary[summary['DM_pvalue'] < 0.05]
print('\n=== DM significant (p < 0.05) ===')
if sig.empty:
    print('Khong co horizon nao DM p<0.05 tren mau test hien tai.')
    print('Ket luan Framework: AR co the thang RMSE nhung chua chac vuot RW ve mat thong ke.')
else:
    display(sig)

=== AR vs RW (RMSE/MAE) ===


,factor,horizon,RMSE_AR,RMSE_RW,AR_better_RMSE,MAE_AR,MAE_RW,MAE_better,DM_pvalue
0,level,1,0.230802,0.228743,False,0.183030,0.181566,False,0.382053
1,level,3,0.415918,0.408680,False,0.321538,0.307709,False,0.491166
2,level,6,0.567871,0.549869,False,0.460723,0.431648,False,0.525729
3,level,12,0.784076,0.760011,False,0.645228,0.620488,False,0.738485
4,slope,1,0.311246,0.308662,False,0.225511,0.226482,True,0.456809
5,slope,3,0.565752,0.555644,False,0.429369,0.415861,False,0.485166
6,slope,6,0.855538,0.837223,False,0.640863,0.643456,True,0.640294
7,slope,12,1.358363,1.346475,False,1.021786,1.060653,True,0.899207
8,curvature,1,0.698008,0.699661,True,0.510038,0.511922,True,0.899548
9,curvature,3,1.164599,1.177612,True,0.880228,0.880443,True,0.766644



=== DM significant (p < 0.05) ===
Khong co horizon nao DM p<0.05 tren mau test hien tai.
Ket luan Framework: AR co the thang RMSE nhung chua chac vuot RW ve mat thong ke.


## 4. Xuat bang cho bao cao

In [5]:
from pathlib import Path
out_path = Path('../data/ar_vs_rw_results.csv')
out_path.parent.mkdir(exist_ok=True)
summary.to_csv(out_path, index=False)
print(f'Da ghi {len(summary)} dong -> {out_path.resolve()}')
summary.head()

Da ghi 12 dong -> D:\Projects\Yield-curve-modeling-and-Forecasting\data\ar_vs_rw_results.csv


,factor,horizon,ar_lag,RMSE_AR,RMSE_RW,MAE_AR,MAE_RW,n_obs,DM_stat,DM_pvalue
0,level,1,1,0.230802,0.228743,0.183030,0.181566,129,0.877141,0.382053
1,level,3,1,0.415918,0.408680,0.321538,0.307709,127,0.690474,0.491166
2,level,6,1,0.567871,0.549869,0.460723,0.431648,124,0.636353,0.525729
3,level,12,1,0.784076,0.760011,0.645228,0.620488,118,0.334655,0.738485
4,slope,1,1,0.311246,0.308662,0.225511,0.226482,129,0.746376,0.456809
